# RSB Pipeline Research Replica

This notebook runs the Official Railway / RSB pipeline in a research-safe way:

1. crawl a small sample from railway zone websites
2. write raw results under `research/output/`
3. optionally run the same `engine.py` AI generation
4. score bilinguality and Hindi output quality

Set `RUN_CRAWL = True` and `RUN_AI = True` only when you want to call live websites / Claude API.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys
from collections import Counter

# Locate the research package by walking up, so this notebook works whether the
# kernel's working directory is research/notebooks/, research/ or the repo root.
def _find_research(start):
    for p in (start, *start.parents):
        for cand in (p, p / 'research'):
            if (cand / 'shared').is_dir() and (cand / 'web_search').is_dir():
                return cand
    raise RuntimeError('cannot locate the research package from %s' % start)

RESEARCH_DIR = _find_research(Path.cwd().resolve())
PROJECT_ROOT = RESEARCH_DIR.parent
# Both are needed: PROJECT_ROOT for the webapp modules (engine, web_agent),
# RESEARCH_DIR for the research packages (web_search, rsb_search, shared).
for _p in (PROJECT_ROOT, RESEARCH_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

OUTPUT_DIR = RESEARCH_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CONFIG_FILE = RESEARCH_DIR / 'hindi_railway_sources.json'
RSB_RAW_FILE = OUTPUT_DIR / 'rsb_crawl_output_research.json'
RSB_AI_FILE = OUTPUT_DIR / 'rsb_agency_output_research.json'
RSB_STATUS_FILE = OUTPUT_DIR / 'rsb_pipeline_status_research.json'

RUN_CRAWL = False       # crawl the Hindi (lang=1) edition via research/rsb_search/pipeline
RUN_AI = False
HINDI_ONLY = False      # True = keep only zones whose BODY is Devanagari (excludes NR)

source_config = json.loads(SOURCE_CONFIG_FILE.read_text(encoding='utf-8'))
RSB_SOURCES = source_config['source_groups']['rsb_sources']['sources']
SELECTED_ZONES = [s['crawler_zone_code'] for s in RSB_SOURCES if s.get('crawler_zone_code')]
SELECTED_ZONES = SELECTED_ZONES[:1]  # small default sample; increase for broader research
PER_ZONE = 2
DATE_FROM = ''  # crawler format: dd-mm-yyyy
DATE_TO = ''    # crawler format: dd-mm-yyyy

print('Project root:', PROJECT_ROOT)
print('Research output:', OUTPUT_DIR)
print('RSB source config:', SOURCE_CONFIG_FILE)
print('Available RSB sources:', [s['name'] for s in RSB_SOURCES])
print('Selected crawler zones:', SELECTED_ZONES)


def load_json_file(path):
    """Read a research JSON file, tolerating missing / empty / truncated files."""
    from pathlib import Path as _P
    p = _P(path)
    if not p.exists() or p.stat().st_size == 0:
        return None
    try:
        return json.loads(p.read_text(encoding='utf-8'))
    except json.JSONDecodeError as e:
        print('WARNING: %s is not valid JSON (%s). Ignoring it.' % (p.name, e))
        return None


## Stage 1: RSB Crawl (Hindi edition)

`config.ZONES` points every zone at `view_section.jsp?lang=0` — the **English** edition.
Measured on Central Railway: `lang=0` gives 18 releases with 0 Hindi titles; `lang=1`
gives the same 18 with 17 Hindi titles and ~100% Devanagari bodies.

`research/rsb_search/pipeline.py` rewrites the URL to `lang=1` and reuses the webapp's own
`crawler._parse_listing` / `_parse_detail`. Coverage varies by zone, so every article
records `body_hindi_ratio` rather than trusting the zone's advertised language.

Writes only to `research/output/`, and dedups within the run — the persistent
`seen_articles.json` is never touched.


In [ ]:
from rsb_search import pipeline as rp
from shared.hindi_patches import apply_hindi_patches

# The webapp's dedup._normalise deletes Devanagari; rebind before any Hindi runs through.
apply_hindi_patches()

if RUN_CRAWL:
    raw_articles = rp.run_rsb_crawl(
        zone_codes=SELECTED_ZONES,
        per_zone=PER_ZONE,
        date_from=DATE_FROM or None,
        date_to=DATE_TO or None,
        hindi_only=HINDI_ONLY,
        out_file=str(RSB_RAW_FILE),
    )
else:
    raw_articles = load_json_file(RSB_RAW_FILE)
    if raw_articles is None:
        raw_articles = []
        print('No RSB research crawl yet. Set RUN_CRAWL = True to fetch.')

print('Raw RSB articles:', len(raw_articles))
if raw_articles:
    from collections import Counter
    print('by zone :', dict(Counter(a['zone_code'] for a in raw_articles)))
    print('Hindi-body articles: %d / %d'
          % (sum(1 for a in raw_articles if a.get('is_hindi_body')), len(raw_articles)))
    lens = sorted(len(a.get('body', '')) for a in raw_articles)
    print('body chars: min=%d median=%d max=%d' % (lens[0], lens[len(lens)//2], lens[-1]))


## Stage 2: AI Generation

This calls the same `engine.run_generate()` used by `POST /generate`, but reads/writes research files.

In [ ]:
import engine
from shared import config_env

engine.STATUS_FILE = str(RSB_STATUS_FILE)

if RUN_AI:
    # shell export > research/.env > project-root .env. See research/config_env.py.
    if not config_env.apply_to_engine(engine):
        print('Generation would return empty results.')
    # engine resumes by title; delete RSB_AI_FILE to force a clean regeneration.
    ai_articles = engine.run_generate(str(RSB_RAW_FILE), str(RSB_AI_FILE))
else:
    ai_articles = load_json_file(RSB_AI_FILE)
    if ai_articles is None:
        ai_articles = raw_articles
        print('No RSB AI output yet; scoring raw crawl instead.')

print('Research AI articles:', len(ai_articles))
print('AI file:', RSB_AI_FILE)


## Stage 3: Hindi And Bilingual Scores

In [ ]:
devanagari_re = re.compile(r'[\u0900-\u097F]')
english_re = re.compile(r'[A-Za-z]')

rows = []
for idx, art in enumerate(ai_articles):
    source_lang = art.get('source_lang', 'unknown')
    title = art.get('title', '') or ''
    body = art.get('body', '') or ''
    story = art.get('ai_news_story', '') or ''
    translated = art.get('ai_news_story_translated', '') or ''

    story_has_hindi = bool(devanagari_re.search(story))
    translation_has_hindi = bool(devanagari_re.search(translated))
    story_has_english = bool(english_re.search(story))
    translation_has_english = bool(english_re.search(translated))
    source_has_hindi = bool(devanagari_re.search(title + ' ' + body))

    direction_ok = False
    if source_lang == 'hi':
        direction_ok = story_has_hindi and translation_has_english
    elif source_lang == 'en':
        direction_ok = story_has_english and translation_has_hindi

    bilingual_score = 0
    bilingual_score += 25 if source_lang in ('hi', 'en') else 0
    bilingual_score += 25 if len(story.strip()) > 30 else 0
    bilingual_score += 25 if len(translated.strip()) > 30 else 0
    bilingual_score += 25 if direction_ok else 0

    hindi_information_score = 0
    hindi_information_score += 35 if source_has_hindi else 0
    hindi_information_score += 35 if story_has_hindi else 0
    hindi_information_score += 30 if translation_has_hindi else 0

    rows.append({
        'idx': idx,
        'zone': art.get('zone_code', ''),
        'date': art.get('date', ''),
        'source_lang': source_lang,
        'bilingual_score': bilingual_score,
        'hindi_information_score': hindi_information_score,
        'fact_score': art.get('fact_score', None),
        'plagiarism_score': art.get('plagiarism_score', None),
        'story_has_hindi': story_has_hindi,
        'translation_has_hindi': translation_has_hindi,
        'title': title[:120],
    })

def avg(values):
    values = [v for v in values if isinstance(v, (int, float))]
    return round(sum(values) / len(values), 2) if values else None

print('Total:', len(rows))
print('Source language counts:', dict(Counter(r['source_lang'] for r in rows)))
print('Average bilingual score:', avg([r['bilingual_score'] for r in rows]))
print('Average Hindi information score:', avg([r['hindi_information_score'] for r in rows]))
print('Average fact score:', avg([r['fact_score'] for r in rows]))
print('Average plagiarism score:', avg([r['plagiarism_score'] for r in rows]))

for row in rows[:25]:
    print(row)

## Stage 4: Grounded Quality Scores

`fact_score`, `impact_level` and `topics` are all produced by the same Claude call that
writes the story — nothing verifies them against the source. Measured over 25 articles,
`fact_score` has only 5 distinct values with 88% in 92-95.

`research/shared/grounding.py` adds deterministic, source-checked fields alongside them:
`coverage_score` (share of source figures the story carried), a hallucination gate,
topic vocabulary + evidence checks, and a rule-derived impact level.


In [ ]:
from shared.score_report import rescore_file
from shared import grounding

scored, scored_path = rescore_file(str(RSB_AI_FILE))
print('scored ->', scored_path)

print('\n%-4s %-6s %-9s %-6s %-7s %s' % ('#','fact','coverage','prec','impact','gates'))
print('-'*72)
for i, a in enumerate(scored):
    print('%-4d %-6s %-9s %-6s %-7s %s' % (
        i, a.get('fact_score'), a.get('coverage_score'), a.get('number_precision'),
        '%s/%s' % (a.get('impact_level'), a.get('impact_rule_level')),
        'OK' if a.get('gates_passed') else '; '.join(a.get('gate_failures', []))[:36]))

_cov = [a['coverage_score'] for a in scored if a.get('coverage_score') is not None]
_fac = [a['fact_score'] for a in scored if a.get('fact_score') is not None]
print('\ndistinct values -> fact_score: %d | coverage_score: %d'
      % (len(set(_fac)), len(set(_cov))))
